<a href="https://colab.research.google.com/github/Manasa2389/-Edge-human-detection-latency-study/blob/main/AI_Decision_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Libraries**

In [3]:
!pip install -q gradio pandas numpy scikit-learn matplotlib plotly openai

**Import Libraries**

In [4]:
import os
import re
import sqlite3
import math
import json
from datetime import datetime, date, timedelta

import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


**Create database**

In [5]:
DB_PATH = "/content/careerpilot.db"

STATUSES = [
    "Saved",
    "Applied",
    "Assessment",
    "Interview",
    "Offer",
    "Rejected",
    "Withdrawn"
]

COMMON_SKILLS = [
    "python",
    "java",
    "javascript",
    "typescript",
    "react",
    "next.js",
    "node.js",
    "fastapi",
    "flask",
    "django",
    "sql",
    "postgresql",
    "mysql",
    "mongodb",
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "keras",
    "machine learning",
    "deep learning",
    "nlp",
    "computer vision",
    "generative ai",
    "llm",
    "rag",
    "openai",
    "aws",
    "azure",
    "gcp",
    "docker",
    "kubernetes",
    "git",
    "github",
    "rest api",
    "data analysis",
    "power bi",
    "tableau",
    "excel",
    "c++",
    "c",
    "arduino",
    "iot",
    "cybersecurity",
    "linux",
    "statistics",
    "data structures",
    "algorithms"
]


def get_connection():
    conn = sqlite3.connect(
        DB_PATH,
        check_same_thread=False
    )
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database():

    conn = get_connection()

    conn.execute("""
    CREATE TABLE IF NOT EXISTS applications (

        id INTEGER PRIMARY KEY AUTOINCREMENT,

        company TEXT NOT NULL,

        role TEXT NOT NULL,

        status TEXT NOT NULL,

        location TEXT,

        application_date TEXT,

        deadline TEXT,

        interview_date TEXT,

        recruiter_email TEXT,

        job_description TEXT,

        notes TEXT,

        created_at TEXT,

        updated_at TEXT
    )
    """)

    conn.commit()
    conn.close()


initialize_database()

print("✅ Database created")

✅ Database created


**Insert demo applications**

In [6]:
def seed_demo_data():

    conn = get_connection()

    count = conn.execute(
        "SELECT COUNT(*) AS count FROM applications"
    ).fetchone()["count"]

    if count > 0:
        conn.close()
        return

    now = datetime.now().isoformat()

    demo_data = [

        (
            "Google",
            "Software Engineering Intern",
            "Applied",
            "Bangalore",
            "2026-08-25",
            "2026-09-15",
            "",
            "",
            """
            Python, Java, data structures, algorithms,
            distributed systems, Git, problem solving.
            """,
            "Prepare DSA questions."
        ),

        (
            "Microsoft",
            "AI/ML Intern",
            "Interview",
            "Hyderabad",
            "2026-08-20",
            "2026-09-12",
            "2026-09-10",
            "",
            """
            Python, machine learning, deep learning,
            NLP, Azure, SQL and data analysis.
            """,
            "Prepare ML fundamentals."
        ),

        (
            "Deloitte",
            "Data Analyst Intern",
            "Assessment",
            "Hyderabad",
            "2026-08-28",
            "2026-09-18",
            "",
            "",
            """
            SQL, Excel, Power BI, Python,
            statistics and data visualization.
            """,
            "Practice SQL joins."
        ),

        (
            "Amazon",
            "SDE Intern",
            "Saved",
            "Bangalore",
            "",
            "2026-09-20",
            "",
            "",
            """
            Java, Python, data structures,
            algorithms, AWS and problem solving.
            """,
            "Tailor resume."
        ),

        (
            "AI Startup",
            "Generative AI Intern",
            "Applied",
            "Remote",
            "2026-09-01",
            "2026-09-11",
            "",
            "",
            """
            Python, LLM, RAG, embeddings,
            FastAPI, vector databases and OpenAI.
            """,
            "High-priority application."
        )

    ]

    for item in demo_data:

        conn.execute("""
        INSERT INTO applications
        (
            company,
            role,
            status,
            location,
            application_date,
            deadline,
            interview_date,
            recruiter_email,
            job_description,
            notes,
            created_at,
            updated_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (*item, now, now))

    conn.commit()
    conn.close()


seed_demo_data()

print("✅ Demo applications added")

✅ Demo applications added


**AI skill extraction**

In [7]:
def clean_text(text):

    if text is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def extract_skills(text):

    text = clean_text(text).lower()

    found = []

    for skill in COMMON_SKILLS:

        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(skill.lower())
            + r"(?![a-z0-9])"
        )

        if re.search(pattern, text):

            found.append(skill)

    return sorted(set(found))


print(
    extract_skills(
        "Python, machine learning, React and SQL"
    )
)

['machine learning', 'python', 'react', 'sql']


**Resume/job matching engine**

In [8]:
DEFAULT_RESUME = """
B.Tech student with strong skills in Python, Java, React,
FastAPI, SQL, machine learning, deep learning, data analysis,
TensorFlow, PyTorch, Git, GitHub, cloud fundamentals,
REST APIs and AI application development.

Built academic and hackathon projects involving
AI/ML, computer vision, IoT and web applications.

Strong problem solving, communication and teamwork skills.
"""


def calculate_match(resume, job_description):

    resume = clean_text(resume)
    job_description = clean_text(job_description)

    if not resume or not job_description:

        return {
            "score": 0,
            "matched": [],
            "missing": [],
            "similarity": 0
        }

    try:

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2)
        )

        vectors = vectorizer.fit_transform(
            [resume, job_description]
        )

        similarity = cosine_similarity(
            vectors[0:1],
            vectors[1:2]
        )[0][0]

    except:

        similarity = 0

    resume_skills = set(
        extract_skills(resume)
    )

    job_skills = set(
        extract_skills(job_description)
    )

    matched = sorted(
        resume_skills.intersection(job_skills)
    )

    missing = sorted(
        job_skills.difference(resume_skills)
    )

    skill_score = (
        len(matched) /
        max(1, len(job_skills))
    )

    final_score = (
        0.55 * similarity +
        0.45 * skill_score
    ) * 100

    return {
        "score": round(final_score, 1),
        "matched": matched,
        "missing": missing,
        "similarity": round(
            similarity * 100,
            1
        )
    }


result = calculate_match(
    DEFAULT_RESUME,
    """
    Python, machine learning, SQL,
    FastAPI and React.
    """
)

result

{'score': np.float64(53.5),
 'matched': ['fastapi', 'machine learning', 'python', 'react', 'sql'],
 'missing': [],
 'similarity': np.float64(15.5)}

**Deadline risk engine**

In [9]:
def deadline_risk(deadline):

    if not deadline:
        return 999, "⚪ No deadline"

    try:

        deadline_date = datetime.strptime(
            deadline,
            "%Y-%m-%d"
        ).date()

        days_remaining = (
            deadline_date - date.today()
        ).days

    except:

        return 999, "⚪ Invalid deadline"

    if days_remaining < 0:

        return (
            days_remaining,
            "🔴 Deadline Passed"
        )

    elif days_remaining <= 2:

        return (
            days_remaining,
            "🚨 CRITICAL"
        )

    elif days_remaining <= 7:

        return (
            days_remaining,
            "🟠 HIGH"
        )

    elif days_remaining <= 14:

        return (
            days_remaining,
            "🟡 MEDIUM"
        )

    else:

        return (
            days_remaining,
            "🟢 LOW"
        )

**Opportunity prediction engine**

In [10]:
def opportunity_score(
    application,
    resume=DEFAULT_RESUME
):

    job = application.get(
        "job_description",
        ""
    )

    match = calculate_match(
        resume,
        job
    )

    match_score = match["score"]

    days, risk = deadline_risk(
        application.get(
            "deadline",
            ""
        )
    )

    if days < 0:

        urgency = 0

    elif days <= 2:

        urgency = 100

    elif days <= 7:

        urgency = 80

    elif days <= 14:

        urgency = 60

    elif days <= 30:

        urgency = 35

    else:

        urgency = 15

    status = application.get(
        "status",
        "Saved"
    )

    status_bonus = {

        "Interview": 20,

        "Assessment": 15,

        "Applied": 8,

        "Offer": 25,

        "Saved": 5,

        "Rejected": 0,

        "Withdrawn": 0

    }.get(status, 0)

    final = (
        0.65 * match_score +
        0.25 * urgency +
        status_bonus
    )

    return min(
        100,
        round(final)
    )


print("✅ Opportunity engine ready")

✅ Opportunity engine ready


**Next best action**

In [11]:
def next_action(row):

    status = row["status"]

    if status == "Interview":

        return "🎤 Prepare for interview"

    if status == "Assessment":

        return "🧪 Complete assessment"

    if status == "Applied":

        return "✉️ Follow up / monitor"

    if status == "Saved":

        return "📝 Tailor resume + apply"

    if status == "Offer":

        return "🏆 Evaluate offer"

    if status == "Rejected":

        return "📚 Analyze gaps"

    return "📌 Monitor"

**Load dashboard data**

In [12]:
def get_applications():

    conn = get_connection()

    df = pd.read_sql_query(
        """
        SELECT *
        FROM applications
        ORDER BY id DESC
        """,
        conn
    )

    conn.close()

    return df


def create_dashboard_data():

    df = get_applications()

    if df.empty:
        return df

    scores = []
    risks = []
    actions = []
    days_left = []

    for _, row in df.iterrows():

        score = opportunity_score(
            row.to_dict()
        )

        days, risk = deadline_risk(
            row["deadline"]
        )

        scores.append(score)
        risks.append(risk)
        actions.append(
            next_action(row)
        )
        days_left.append(days)

    df["Opportunity Score"] = scores

    df["Deadline Risk"] = risks

    df["Days Left"] = days_left

    df["Next Best Action"] = actions

    return df.sort_values(
        "Opportunity Score",
        ascending=False
    )


df = create_dashboard_data()

df[
    [
        "company",
        "role",
        "status",
        "Opportunity Score",
        "Deadline Risk",
        "Next Best Action"
    ]
]

,company,role,status,Opportunity Score,Deadline Risk,Next Best Action
3,Microsoft,AI/ML Intern,Interview,68,🟠 HIGH,🎤 Prepare for interview
4,Google,Software Engineering Intern,Applied,49,🟠 HIGH,✉️ Follow up / monitor
2,Deloitte,Data Analyst Intern,Assessment,43,🟡 MEDIUM,🧪 Complete assessment
0,AI Startup,Generative AI Intern,Applied,41,🟠 HIGH,✉️ Follow up / monitor
1,Amazon,SDE Intern,Saved,35,🟡 MEDIUM,📝 Tailor resume + apply


**Analytics**

In [13]:
def analytics_summary():

    df = create_dashboard_data()

    if df.empty:

        return "No applications available."

    total = len(df)

    active = len(
        df[
            df["status"].isin(
                [
                    "Applied",
                    "Assessment",
                    "Interview",
                    "Offer"
                ]
            )
        ]
    )

    interviews = len(
        df[df["status"] == "Interview"]
    )

    offers = len(
        df[df["status"] == "Offer"]
    )

    urgent = len(
        df[
            df["Deadline Risk"].isin(
                [
                    "🚨 CRITICAL",
                    "🟠 HIGH"
                ]
            )
        ]
    )

    average_score = round(
        df["Opportunity Score"].mean(),
        1
    )

    top = df.iloc[0]

    return f"""
# 📊 CareerPilot Intelligence

### Application Metrics

| Metric | Value |
|---|---:|
| Total Applications | **{total}** |
| Active Applications | **{active}** |
| Interviews | **{interviews}** |
| Offers | **{offers}** |
| Urgent Applications | **{urgent}** |
| Average Opportunity Score | **{average_score}/100** |

---

## 🏆 Highest Priority Opportunity

**{top['company']}**

### {top['role']}

Opportunity Score: **{top['Opportunity Score']}/100**

Next Action:

**{top['Next Best Action']}**
"""

**Add applications**

In [14]:
def add_application(
    company,
    role,
    status,
    location,
    application_date,
    deadline,
    interview_date,
    recruiter_email,
    job_description,
    notes
):

    if not company or not role:

        return (
            create_dashboard_data(),
            "❌ Company and role are required."
        )

    now = datetime.now().isoformat()

    conn = get_connection()

    conn.execute(
        """
        INSERT INTO applications
        (
            company,
            role,
            status,
            location,
            application_date,
            deadline,
            interview_date,
            recruiter_email,
            job_description,
            notes,
            created_at,
            updated_at
        )

        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            company,
            role,
            status,
            location,
            application_date,
            deadline,
            interview_date,
            recruiter_email,
            job_description,
            notes,
            now,
            now
        )
    )

    conn.commit()
    conn.close()

    return (
        create_dashboard_data(),
        f"## ✅ Added\n\n**{company} — {role}**"
    )

**Update status**

In [15]:
def update_application_status(
    application_id,
    new_status
):

    try:

        application_id = int(
            application_id
        )

    except:

        return (
            create_dashboard_data(),
            "❌ Invalid application ID."
        )

    conn = get_connection()

    cursor = conn.execute(
        """
        UPDATE applications

        SET status = ?,
            updated_at = ?

        WHERE id = ?
        """,
        (
            new_status,
            datetime.now().isoformat(),
            application_id
        )
    )

    conn.commit()

    changed = cursor.rowcount

    conn.close()

    if changed:

        message = "✅ Status updated."

    else:

        message = "⚠️ Application ID not found."

    return (
        create_dashboard_data(),
        message
    )

**Delete application**

In [16]:
def delete_application(application_id):

    try:

        application_id = int(
            application_id
        )

    except:

        return (
            create_dashboard_data(),
            "❌ Invalid ID."
        )

    conn = get_connection()

    cursor = conn.execute(
        """
        DELETE FROM applications
        WHERE id = ?
        """,
        (application_id,)
    )

    conn.commit()

    deleted = cursor.rowcount

    conn.close()

    if deleted:

        message = "🗑️ Application deleted."

    else:

        message = "⚠️ Application not found."

    return (
        create_dashboard_data(),
        message
    )

**Resume matcher UI function**

In [17]:
def analyze_resume_job(
    resume,
    job_description
):

    result = calculate_match(
        resume,
        job_description
    )

    score = result["score"]

    if score >= 80:

        verdict = "🔥 Excellent Match"

    elif score >= 65:

        verdict = "🟢 Strong Match"

    elif score >= 50:

        verdict = "🟡 Moderate Match"

    else:

        verdict = "🔴 Low Match"

    matched = ", ".join(
        result["matched"]
    ) or "None detected"

    missing = ", ".join(
        result["missing"]
    ) or "No major gaps detected"

    return f"""
# 🎯 AI Resume Match

## Match Score

# **{score}/100**

### {verdict}

---

### ✅ Matching Skills

{matched}

---

### ⚠️ Skill Gaps

{missing}

---

### 📊 Semantic Similarity

**{result['similarity']}%**

### 💡 Recommendation

{
    "Apply immediately and customize the resume."
    if score >= 65
    else
    "Improve missing skills before targeting similar roles."
}
"""

**Follow-up email generator**

In [18]:
def generate_followup_email(
    company,
    role,
    status,
    recruiter_email,
    notes
):

    return f"""
# ✉️ Follow-up Email

**Subject:** Follow-up regarding {role} application

Dear Hiring Team,

I hope you are doing well.

I am writing to politely follow up regarding my application for the
**{role}** position at **{company}**.

I remain very interested in the opportunity and would be grateful
for any update regarding the application process.

Current application status: **{status}**

{notes}

Please let me know if any additional information is required from
my side.

Thank you for your time and consideration.

Best regards,

**[Your Name]**
"""

**Interview Coach**

In [19]:
def interview_coach(
    company,
    role,
    job_description
):

    skills = extract_skills(
        job_description
    )

    skills_text = ", ".join(
        skills[:12]
    )

    return f"""
# 🎤 AI Interview Coach

## Company

**{company}**

## Position

**{role}**

---

# 🔥 Technical Questions

### 1.
Explain your strongest project and your contribution.

### 2.
Explain the difference between supervised
and unsupervised machine learning.

### 3.
What is the time complexity of common
data structures and algorithms?

### 4.
How would you design a REST API?

### 5.
How would you debug a production issue?

---

# 🧠 Role-Specific Skills

{skills_text}

---

# 👨‍💻 Behavioral Questions

### Tell me about yourself.

### Why do you want this internship?

### Tell me about a difficult project.

### How did you solve a technical problem?

### Where do you see yourself in five years?

---

# 🎯 60-Second Introduction

Hello, I am a B.Tech student with a strong interest
in software development and artificial intelligence.

I enjoy building practical projects that combine
programming, data and intelligent automation.

I have experience working with Python and modern
development technologies and have worked on academic
and hackathon projects.

I am particularly interested in this opportunity
because it matches my technical interests and gives
me an opportunity to learn from an experienced team.

Thank you for the opportunity.
"""

**Export CSV**

In [20]:
def export_applications():

    path = "/content/careerpilot_applications.csv"

    df = create_dashboard_data()

    df.to_csv(
        path,
        index=False
    )

    return path

**Charts**

In [21]:
def status_chart():

    df = get_applications()

    if df.empty:
        return None

    counts = (
        df["status"]
        .value_counts()
        .reset_index()
    )

    counts.columns = [
        "Status",
        "Applications"
    ]

    fig = px.bar(
        counts,
        x="Status",
        y="Applications",
        title="Application Pipeline"
    )

    return fig


def score_chart():

    df = create_dashboard_data()

    if df.empty:
        return None

    top = df.head(10).copy()

    top["Company Role"] = (
        top["company"] +
        " — " +
        top["role"]
    )

    fig = px.bar(
        top,
        x="Company Role",
        y="Opportunity Score",
        title="Top Opportunity Scores"
    )

    fig.update_layout(
        xaxis_tickangle=-45
    )

    return fig

**Build the complete application**

In [24]:
with gr.Blocks(
    title="CareerPilot AI"
) as app:

    gr.Markdown(
        """
# 🚀 CareerPilot AI

## AI-Powered Internship Application Intelligence Platform

### **Track → Analyze → Prioritize → Act → Win**

> CareerPilot doesn't just store applications.
> It tells you **which opportunity deserves your attention next.**
        """
    )

    # -----------------------------------
    # DASHBOARD
    # -----------------------------------

    with gr.Tab("📊 Dashboard"):

        summary = gr.Markdown(
            analytics_summary()
        )

        refresh_button = gr.Button(
            "🔄 Refresh Dashboard"
        )

        applications_table = gr.Dataframe(
            value=create_dashboard_data(),
            interactive=False,
            wrap=True
        )

        with gr.Row():

            status_plot = gr.Plot(
                value=status_chart()
            )

            score_plot = gr.Plot(
                value=score_chart()
            )

        def refresh_dashboard():

            return (
                analytics_summary(),
                create_dashboard_data(),
                status_chart(),
                score_chart()
            )

        refresh_button.click(
            refresh_dashboard,
            outputs=[
                summary,
                applications_table,
                status_plot,
                score_plot
            ]
        )

    # -----------------------------------
    # ADD APPLICATION
    # -----------------------------------

    with gr.Tab("➕ Add Application"):

        with gr.Row():

            company = gr.Textbox(
                label="Company",
                placeholder="Google"
            )

            role = gr.Textbox(
                label="Role",
                placeholder="AI/ML Intern"
            )

        with gr.Row():

            status = gr.Dropdown(
                STATUSES,
                value="Applied",
                label="Status"
            )

            location = gr.Textbox(
                label="Location",
                placeholder="Hyderabad"
            )

        with gr.Row():

            application_date = gr.Textbox(
                label="Application Date",
                placeholder="YYYY-MM-DD"
            )

            deadline = gr.Textbox(
                label="Deadline",
                placeholder="YYYY-MM-DD"
            )

            interview_date = gr.Textbox(
                label="Interview Date",
                placeholder="YYYY-MM-DD"
            )

        recruiter_email = gr.Textbox(
            label="Recruiter Email"
        )

        job_description = gr.Textbox(
            label="Job Description / Required Skills",
            lines=8
        )

        notes = gr.Textbox(
            label="Notes",
            lines=4
        )

        add_button = gr.Button(
            "🚀 Add Application",
            variant="primary"
        )

        add_result = gr.Markdown()

        add_button.click(
            add_application,
            inputs=[
                company,
                role,
                status,
                location,
                application_date,
                deadline,
                interview_date,
                recruiter_email,
                job_description,
                notes
            ],
            outputs=[
                applications_table,
                add_result
            ]
        )

    # -----------------------------------
    # RESUME MATCHER
    # -----------------------------------

    with gr.Tab("🎯 AI Resume Matcher"):

        resume_input = gr.Textbox(
            value=DEFAULT_RESUME,
            label="Resume",
            lines=14
        )

        job_input = gr.Textbox(
            label="Paste Job Description",
            lines=14
        )

        match_button = gr.Button(
            "🧠 Analyze Match",
            variant="primary"
        )

        match_output = gr.Markdown()

        match_button.click(
            analyze_resume_job,
            inputs=[
                resume_input,
                job_input
            ],
            outputs=match_output
        )

    # -----------------------------------
    # FOLLOW-UP
    # -----------------------------------

    with gr.Tab("✉️ Follow-up Generator"):

        with gr.Row():

            follow_company = gr.Textbox(
                label="Company"
            )

            follow_role = gr.Textbox(
                label="Role"
            )

        with gr.Row():

            follow_status = gr.Dropdown(
                STATUSES,
                value="Applied",
                label="Status"
            )

            follow_email = gr.Textbox(
                label="Recruiter Email"
            )

        follow_notes = gr.Textbox(
            label="Additional Notes",
            lines=4
        )

        follow_button = gr.Button(
            "✉️ Generate Email",
            variant="primary"
        )

        follow_output = gr.Markdown()

        follow_button.click(
            generate_followup_email,
            inputs=[
                follow_company,
                follow_role,
                follow_status,
                follow_email,
                follow_notes
            ],
            outputs=follow_output
        )

    # -----------------------------------
    # INTERVIEW COACH
    # -----------------------------------

    with gr.Tab("🎤 Interview Coach"):

        interview_company = gr.Textbox(
            label="Company"
        )

        interview_role = gr.Textbox(
            label="Role"
        )

        interview_job = gr.Textbox(
            label="Job Description",
            lines=10
        )

        interview_button = gr.Button(
            "🎤 Generate Interview Plan",
            variant="primary"
        )

        interview_output = gr.Markdown()

        interview_button.click(
            interview_coach,
            inputs=[
                interview_company,
                interview_role,
                interview_job
            ],
            outputs=interview_output
        )

    # -----------------------------------
    # APPLICATION MANAGEMENT
    # -----------------------------------

    with gr.Tab("⚙️ Manage Applications"):

        gr.Markdown(
            "## Update Application Status"
        )

        with gr.Row():

            update_id = gr.Number(
                label="Application ID",
                precision=0
            )

            update_status = gr.Dropdown(
                STATUSES,
                value="Applied",
                label="New Status"
            )

        update_button = gr.Button(
            "🔄 Update Status"
        )

        update_result = gr.Markdown()

        update_button.click(
            update_application_status,
            inputs=[
                update_id,
                update_status
            ],
            outputs=[
                applications_table,
                update_result
            ]
        )

        gr.Markdown(
            "## Delete Application"
        )

        delete_id = gr.Number(
            label="Application ID",
            precision=0
        )

        delete_button = gr.Button(
            "🗑️ Delete"
        )

        delete_result = gr.Markdown()

        delete_button.click(
            delete_application,
            inputs=delete_id,
            outputs=[
                applications_table,
                delete_result
            ]
        )

        gr.Markdown(
            "## Export Data"
        )

        export_button = gr.Button(
            "📥 Export CSV"
        )

        export_file = gr.File()

        export_button.click(
            export_applications,
            outputs=export_file
        )

    gr.Markdown(
        """
        """
    )
